# Inventory Management Simulation — Interactive Notebook

This notebook provides an interactive walkthrough of the simulation framework:
1. Load products and configuration
2. Run a single product under all three policies
3. Compare policies with metrics and charts
4. Run scenario analysis (high demand, long lead times, stress test)
5. Multi-product summary heatmap


In [ ]:
import sys
sys.path.insert(0, '..')  # Make sure root package is importable

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from simulation import (
    Product, InventoryModel, DemandGenerator,
    ReorderPointPolicy, MinMaxPolicy, PeriodicReviewPolicy,
)
from analysis import compute_metrics, compare_policies, InventoryCharts
from analysis.metrics import rank_policies

print('✅ Imports OK')

## 1. Load Configuration and Products

In [ ]:
with open('../data/config.json') as f:
    cfg = json.load(f)

products_df = pd.read_csv('../data/products.csv')
products = [Product.from_dict(row) for row in products_df.to_dict(orient='records')]

print(f'Loaded {len(products)} products:')
products_df[['product_id', 'product_name', 'mean_demand', 'demand_pattern', 'reorder_point', 'max_stock']]

## 2. Single Product — All Three Policies (Baseline)

In [ ]:
# Choose product
product = products[0]  # P001 — Electronic Components
SEED = 42
NUM_DAYS = 365

policies = {
    'Reorder Point (s,Q)': ReorderPointPolicy(
        reorder_point=product.reorder_point,
        order_quantity=product.order_quantity,
    ),
    'Min-Max (s,S)': MinMaxPolicy(
        min_level=product.reorder_point,
        max_level=product.max_stock,
    ),
    'Periodic Review (R,S)': PeriodicReviewPolicy(
        review_period=product.review_period,
        target_level=product.max_stock,
    ),
}

results = {}
for name, policy in policies.items():
    dg = DemandGenerator(
        pattern=product.demand_pattern,
        mean=product.mean_demand,
        std=product.std_demand,
        seed=SEED,
    )
    model = InventoryModel(
        product=product, policy=policy,
        demand_generator=dg, num_days=NUM_DAYS, seed=SEED,
    )
    results[name] = model.results_df()
    print(f'  ✓ {name}')

print('\nSimulation complete.')

## 3. Policy Metrics Comparison

In [ ]:
comparison = compare_policies(results, product_name=product.product_id)
display_cols = [
    'total_cost', 'total_holding_cost', 'total_stockout_cost', 'total_ordering_cost',
    'fill_rate_pct', 'service_level_pct', 'stockout_days', 'num_orders_placed', 'avg_inventory'
]
comparison[display_cols].style.format('{:.2f}').background_gradient(cmap='RdYlGn_r', subset=['total_cost']).background_gradient(cmap='RdYlGn', subset=['fill_rate_pct', 'service_level_pct'])

In [ ]:
ranked = rank_policies(comparison)
print('Policy Rankings (lower composite_score = better overall):')
ranked[['total_cost', 'fill_rate_pct', 'cost_rank', 'fill_rank', 'composite_score']]

## 4. Inventory Level Charts

In [ ]:
charts = InventoryCharts(output_dir='../outputs/charts')

for policy_name, df in results.items():
    print(f'Plotting: {policy_name}')
    charts.plot_inventory_levels(
        df,
        policy_name=policy_name,
        product_name=product.product_id,
        reorder_point=product.reorder_point,
    )

In [ ]:
# Policy comparison chart
charts.plot_policy_comparison(
    comparison.reset_index(),
    product_name=product.product_id,
)

## 5. Scenario Analysis

In [ ]:
scenarios = cfg['scenarios']
scenario_results = {}
best_policy_name = ranked.index[0]  # Use best-ranked policy

print(f'Running scenario analysis with best policy: {best_policy_name}\n')

for sc_name, sc_cfg in scenarios.items():
    policy = policies[best_policy_name]
    dg = DemandGenerator(
        pattern=product.demand_pattern,
        mean=product.mean_demand,
        std=product.std_demand,
        seed=SEED,
        demand_multiplier=sc_cfg['demand_multiplier'],
    )
    model = InventoryModel(
        product=product, policy=policy,
        demand_generator=dg, num_days=NUM_DAYS,
        lead_time_multiplier=sc_cfg['lead_time_multiplier'],
        seed=SEED,
    )
    sc_df = model.results_df()
    sc_m = compute_metrics(sc_df, policy_name=best_policy_name, product_name=product.product_id)
    scenario_results[sc_name] = sc_df
    print(f'  {sc_name:20s}  cost=${sc_m["total_cost"]:>10,.2f}  fill={sc_m["fill_rate_pct"]:.1f}%  stockouts={sc_m["stockout_days"]} days')

In [ ]:
# Scenario comparison chart
all_sc_metrics = {
    sc: pd.DataFrame([compute_metrics(df)]) for sc, df in scenario_results.items()
}
charts.plot_scenario_comparison(all_sc_metrics, product_name=product.product_id)

## 6. Demand Pattern Exploration

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
fig.suptitle('Demand Patterns — 365 Day Preview', fontsize=13, fontweight='bold')

patterns = ['normal', 'poisson', 'seasonal', 'sporadic', 'constant']
colors = ['#2563EB', '#10B981', '#F59E0B', '#EF4444', '#8B5CF6']

for i, (pattern, color) in enumerate(zip(patterns, colors)):
    ax = axes[i // 3][i % 3]
    dg = DemandGenerator(pattern=pattern, mean=20, std=5, seed=42)
    series = dg.generate_series(365)
    ax.plot(series, color=color, lw=0.8, alpha=0.85)
    ax.axhline(series.mean(), color='gray', ls='--', lw=1, label=f'Mean={series.mean():.1f}')
    ax.set_title(pattern.capitalize())
    ax.set_xlabel('Day')
    ax.set_ylabel('Demand')
    ax.legend(fontsize=8)

axes[1][2].axis('off')  # Hide unused subplot
plt.tight_layout()
plt.savefig('../outputs/charts/demand_patterns_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved demand_patterns_overview.png')

## 7. All Products — Full Run and Heatmap

In [ ]:
all_metrics = []

for prod in products:
    prod_policies = {
        'Reorder Point (s,Q)': ReorderPointPolicy(
            reorder_point=prod.reorder_point,
            order_quantity=prod.order_quantity,
        ),
        'Min-Max (s,S)': MinMaxPolicy(
            min_level=prod.reorder_point,
            max_level=prod.max_stock,
        ),
        'Periodic Review (R,S)': PeriodicReviewPolicy(
            review_period=prod.review_period,
            target_level=prod.max_stock,
        ),
    }
    for pol_name, pol in prod_policies.items():
        dg = DemandGenerator(
            pattern=prod.demand_pattern,
            mean=prod.mean_demand,
            std=prod.std_demand,
            seed=SEED,
        )
        model = InventoryModel(
            product=prod, policy=pol,
            demand_generator=dg, num_days=NUM_DAYS, seed=SEED,
        )
        df = model.results_df()
        m = compute_metrics(df, policy_name=pol_name, product_name=prod.product_id)
        all_metrics.append(m)
    print(f'  ✓ {prod.product_id}')

all_metrics_df = pd.DataFrame(all_metrics)
print(f'\nTotal runs: {len(all_metrics_df)}')

In [ ]:
# Summary heatmap: total cost across all products and policies
charts.plot_summary_heatmap(all_metrics_df, metric='total_cost')
charts.plot_summary_heatmap(all_metrics_df, metric='fill_rate_pct')
charts.plot_summary_heatmap(all_metrics_df, metric='stockout_days')
print('Heatmaps saved!')

In [ ]:
# Final summary table
pivot = all_metrics_df.pivot_table(
    index='product', columns='policy',
    values=['total_cost', 'fill_rate_pct'], aggfunc='mean'
)
pivot.round(2)